In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Mejos"
STUDENT_ID = "TUPM-26-3605"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)

np.set_printoptions(precision=4, suppress=True)

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

raw_measurements = np.genfromtxt(DATA_DIR / "secom.data", dtype=float)
labels = np.loadtxt(DATA_DIR / "secom_labels.data", usecols=0, dtype=int)

# Missing, constant, and usable features (same method as Task 9)
missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)
all_missing = observed == 0
observed_min = np.where(missing, np.inf, raw_measurements).min(axis=0)
observed_max = np.where(missing, -np.inf, raw_measurements).max(axis=0)
constant = (observed > 0) & (observed_min == observed_max)
usable_indices = np.flatnonzero(~(all_missing | constant))

n_features = raw_measurements.shape[1]
feature_names = [f"Feature_{i:03d}" for i in range(n_features)]
retained_features = [feature_names[i] for i in usable_indices]
raw_usable = raw_measurements[:, usable_indices].copy()

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT, "| ID Sum:", ID_SUM, "| Random Seed:", RANDOM_SEED)
print("Usable features:", len(retained_features))

Student: Mejos TUPM-26-3605
Seed Digit: 5 | ID Sum: 22 | Random Seed: 263605
Usable features: 474


In [2]:
# 1. Labeled DataFrame (the index keeps the original SECOM record IDs)
df = pd.DataFrame(raw_measurements.copy(), columns=feature_names)
df["Result"] = labels
df["Outcome"] = np.where(labels == -1, "Pass", "Fail")

# 2. Reproducible balanced sample
pass_positions = np.flatnonzero(labels == -1)
fail_positions = np.flatnonzero(labels == 1)
per_group = min(20 + ID_SUM, len(pass_positions), len(fail_positions))

rng = np.random.default_rng(RANDOM_SEED)
chosen_pass = rng.choice(pass_positions, size=per_group, replace=False)
chosen_fail = rng.choice(fail_positions, size=per_group, replace=False)
row_positions = list(chosen_pass) + list(chosen_fail)

# 3. Five usable features, median imputation with medians from the full dataset
feature_rng = np.random.default_rng(RANDOM_SEED)
feature_positions = feature_rng.choice(len(retained_features), size=5, replace=False)
selected = [retained_features[i] for i in feature_positions]

full_medians = df[selected].median()
sample = df.iloc[row_positions][selected + ["Result", "Outcome"]].copy()
sample[selected] = sample[selected].fillna(full_medians)

# 4. Grouped statistics
grouped = sample.groupby("Outcome")[selected].agg(["count", "mean", "median", "std"])
means = sample.groupby("Outcome")[selected].mean()
sds = sample.groupby("Outcome")[selected].std()

# 5. Absolute and standardized mean difference (pooled within-group sample SD)
abs_diff = (means.loc["Fail"] - means.loc["Pass"]).abs()
pooled_sd = np.sqrt(
    ((per_group - 1) * sds.loc["Pass"] ** 2 + (per_group - 1) * sds.loc["Fail"] ** 2)
    / (2 * per_group - 2)
)
safe_sd = np.where(pooled_sd > 0, pooled_sd, 1.0)
abs_smd = np.where(pooled_sd > 0, abs_diff / safe_sd, 0.0)

# 6. Rank and export
comparison = pd.DataFrame({
    "Feature": selected,
    "Pass_Mean": means.loc["Pass"].to_numpy(),
    "Fail_Mean": means.loc["Fail"].to_numpy(),
    "Abs_Mean_Diff": abs_diff.to_numpy(),
    "Pooled_SD": pooled_sd.to_numpy(),
    "Abs_Std_Mean_Diff": abs_smd
}).sort_values(by="Abs_Std_Mean_Diff", ascending=False)

balanced = sample.copy()
balanced.insert(0, "Example_ID", balanced.index.to_numpy())
balanced.to_csv(OUTPUT_DIR / "balanced_sample.csv", index=False)
comparison.to_csv(OUTPUT_DIR / "feature_comparison.csv", index=False)

# NumPy check of the Pandas group statistics
pass_values = sample.loc[sample["Outcome"] == "Pass", selected].to_numpy()
fail_values = sample.loc[sample["Outcome"] == "Fail", selected].to_numpy()
stats_match = (
    np.allclose(pass_values.mean(axis=0), means.loc["Pass"].to_numpy())
    and np.allclose(fail_values.mean(axis=0), means.loc["Fail"].to_numpy())
    and np.allclose(pass_values.std(axis=0, ddof=1), sds.loc["Pass"].to_numpy())
    and np.allclose(fail_values.std(axis=0, ddof=1), sds.loc["Fail"].to_numpy())
)

print("Per-outcome size:", per_group)
print("Balanced Sample Size:", len(sample))
print(sample["Outcome"].value_counts())
print("Selected Features:", selected)
print("Grouped Statistics:\n", grouped.round(4))
print("Feature Comparison:\n", comparison.round(4))
print("Highest-Ranked Feature:", comparison.iloc[0]["Feature"])
print("Absolute Standardized Mean Difference:", f"{comparison.iloc[0]['Abs_Std_Mean_Diff']:.4f}")
print("NumPy-Pandas Statistics Match:", stats_match)

Per-outcome size: 42
Balanced Sample Size: 84
Outcome
Pass    42
Fail    42
Name: count, dtype: int64
Selected Features: ['Feature_062', 'Feature_285', 'Feature_545', 'Feature_025', 'Feature_337']
Grouped Statistics:
         Feature_062                             Feature_285                  \
              count      mean    median     std       count    mean  median   
Outcome                                                                       
Fail             42  116.3723  115.2954  7.0906          42  2.3037  2.0234   
Pass             42  116.0578  116.3227  7.0288          42  2.1785  1.9298   

                Feature_545                        Feature_025          \
            std       count    mean median     std       count    mean   
Outcome                                                                  
Fail     0.9131          42  7.6124  7.116  1.0598          42  1.2267   
Pass     0.9680          42  7.6077  7.116  1.1105          42  1.2304   

               